

**Please make sure you use "human_env" kernel**



---

### Manipulate VCF file with BCFtools and VCFtools

You can install BCFtools with conda

`conda install bcftools`

You can install VCFtools with conda

`conda install vcftools`


**Check what we get from last session**

View only header

In [1]:
conda install bcftools -y

Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [2]:
conda install vcftools

Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [3]:
!bcftools view -h ./output/haplotypecaller/A.g.vcf

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">
##FORMAT=<ID=PID,Number=1,Type

View only data

In [4]:
!bcftools view -H ./output/haplotypecaller/A.g.vcf | head -10

chr14	1	.	N	<NON_REF>	.	.	END=18601020	GT:DP:GQ:MIN_DP:PL	0/0:0:0:0:0,0,0
chr14	18601021	.	C	<NON_REF>	.	.	END=18601022	GT:DP:GQ:MIN_DP:PL	0/0:1:3:1:0,3,37
chr14	18601023	.	T	<NON_REF>	.	.	END=18601027	GT:DP:GQ:MIN_DP:PL	0/0:2:6:2:0,6,74
chr14	18601028	.	A	<NON_REF>	.	.	END=18601034	GT:DP:GQ:MIN_DP:PL	0/0:3:9:3:0,9,107
chr14	18601035	.	T	<NON_REF>	.	.	END=18601039	GT:DP:GQ:MIN_DP:PL	0/0:4:12:4:0,12,154
chr14	18601040	.	A	<NON_REF>	.	.	END=18601045	GT:DP:GQ:MIN_DP:PL	0/0:7:21:7:0,21,257
chr14	18601046	.	G	<NON_REF>	.	.	END=18601046	GT:DP:GQ:MIN_DP:PL	0/0:9:27:9:0,27,350
chr14	18601047	.	A	<NON_REF>	.	.	END=18601050	GT:DP:GQ:MIN_DP:PL	0/0:10:30:10:0,30,367
chr14	18601051	.	C	<NON_REF>	.	.	END=18601052	GT:DP:GQ:MIN_DP:PL	0/0:11:33:11:0,33,408
chr14	18601053	.	T	<NON_REF>	.	.	END=18601056	GT:DP:GQ:MIN_DP:PL	0/0:12:36:12:0,36,440
[main_vcfview] Error: cannot write to (null)


---
# GenomicDBImport

Combine multiple GVCF file



In [5]:
!gatk GenomicsDBImport -h

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport -h
USAGE: GenomicsDBImport [arguments]

Import VCFs to GenomicsDB
Version:4.6.2.0


Required Arguments:

--genomicsdb-update-workspace-path <String>
                              Workspace when updating GenomicsDB. Can be a POSIX file system absolute or relative path
                              or a HDFS/GCS URL. Use this argument when adding new samples to an existing GenomicsDB
                              workspace or when using the output-interval-list-to-file option. Either this or
                              genomicsdb-workspace-path must be specified. Must point to an existing workspace. 


---
### Prepare sample name mapping file

It is a tab delimited file with 2 column

sampleID|path/to/sample.g.vcf

---

Use a combination of `ls` `xargs` `basename` `awk` to generate list file contain sample name.

`ls` will list all specifiy target

In [6]:
!ls ./output/haplotypecaller/*.g.vcf

./output/haplotypecaller/A.g.vcf


---
A pipe character `|` has been use to conect two command

`xargs` will read standard input delimited by blanks or new line and executes the command.

`-n1` was use to limit 1 argument per command line

`basename`  take phrase path as input then returns the component
       following the final '/'.

In [7]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename

A.g.vcf


Use `awk` a text processing command to split file name.

Chain all commands with pipe `|` and save the result in text file.

In [8]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename \
    | awk '{split($0,a,".");print a[1]}' > ./output/haplotypecaller/input_name.txt

In [9]:
!cat ./output/haplotypecaller/input_name.txt

A


---
Use `realpath` command to create list file contain full path of target file

In [10]:
!realpath ./output/haplotypecaller/*.g.vcf > ./output/haplotypecaller/input_path.txt

In [11]:
!cat ./output/haplotypecaller/input_path.txt

/home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output/haplotypecaller/A.g.vcf


Use `paste` command for join files horizontally.
Use `-d` option to set delimeter

In [12]:
!paste ./output/haplotypecaller/input_name.txt \
    ./output/haplotypecaller/input_path.txt  > ./output/haplotypecaller/samples_map.txt

In [13]:
!cat ./output/haplotypecaller/samples_map.txt

A	/home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output/haplotypecaller/A.g.vcf


---
**Create gennomeDB**

In [14]:
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./output/haplotypecaller/samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xms8G -Xmx16G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport --genomicsdb-workspace-path ./output/genomeDB --batch-size 50 -L chr14 --sample-name-map ./output/haplotypecaller/samples_map.txt --tmp-dir ./tmp --reader-threads 2
14:04:51.550 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
14:04:51.633 INFO  GenomicsDBImport - ------------------------------------------------------------
14:04:51.635 INFO  GenomicsDBImport - The Genome Analysis Toolkit (GATK) v4.6.2.0
14:04:51.635 INFO  Genomics

---
**(Additional) Update existing genomeDB**

You can add new sample to existing genomeDB by using the same command but with different option.

`--genomicsdb-update-workspace-path`

In [ ]:
"""
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-update-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./ouptut/haplotypecaller/new_samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2
"""

---
# GenotypeGVCFS

Perform joint genotyping on single or multiple sample pre-called with HaplotypeCaller

**Single sample**

In [15]:
!gatk GenotypeGVCFs -h

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -h
USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible


In [16]:
!mkdir -p ./output/genotypegvcf

In [17]:
!gatk GenotypeGVCFs -h

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -h
USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible


In [18]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V ./output/haplotypecaller/A.g.vcf \
    -O ./output/genotypegvcf/A.vcf \
    --tmp-dir ./tmp

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V ./output/haplotypecaller/A.g.vcf -O ./output/genotypegvcf/A.vcf --tmp-dir ./tmp
14:09:56.873 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
14:09:56.957 INFO  GenotypeGVCFs - ------------------------------------------------------------
14:09:56.959 INFO  GenotypeGVCFs - The Genome Analysis Toolkit (GATK) v4.6.2.0
14:09:56.959 INFO  GenotypeGVCFs - For support and documentation go to https://software.broadinstitute

In [19]:
!bcftools view -H ./output/genotypegvcf/A.vcf | head -10

chr14	18601430	.	A	C	857.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12.9;ReadPosRankSum=0.036;SOR=5.417	GT:AD:DP:GQ:PL	0/1:19,14:33:99:433,0,524
chr14	18601844	.	T	C	184.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-2.245;DP=69;ExcessHet=0;FS=51.431;MLEAC=1;MLEAF=0.

**Multiple sample**

In [20]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V gendb://${PWD}/output/genomeDB \
    -O ./output/genotypegvcf/combine.vcf \
    --tmp-dir ./tmp

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V gendb:///home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output/genomeDB -O ./output/genotypegvcf/combine.vcf --tmp-dir ./tmp
14:11:53.436 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
14:11:53.528 INFO  GenotypeGVCFs - ------------------------------------------------------------
14:11:53.530 INFO  GenotypeGVCFs - The Gen


**View VCF file**

In [21]:
!bcftools view -h ./output/genotypegvcf/combine.vcf | tail

##INFO=<ID=RAW_MQandDP,Number=2,Type=Integer,Description="Raw data (sum of squared MQ and total depth) for improved RMS Mapping Quality calculation. Incompatible with deprecated RAW_MQ formulation.">
##INFO=<ID=ReadPosRankSum,Number=1,Type=Float,Description="Z-score from Wilcoxon rank sum test of Alt vs. Ref read position bias">
##INFO=<ID=SOR,Number=1,Type=Float,Description="Symmetric Odds Ratio of 2x2 contingency table to detect strand bias">
##contig=<ID=chr14,length=107043718>
##source=GenomicsDBImport
##source=GenotypeGVCFs
##source=HaplotypeCaller
##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output/genotypegvcf/combine.vcf; Date=Fri Apr 24 14:12:36 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	A


In [22]:
!bcftools view -H ./output/genotypegvcf/combine.vcf | head -10

chr14	18601430	.	A	C	857.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12.9;ReadPosRankSum=0.036;SOR=5.417	GT:AD:DP:GQ:PL	0/1:19,14:33:99:433,0,524
chr14	18601844	.	T	C	184.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-2.245;DP=69;ExcessHet=0;FS=51.431;MLEAC=1;MLEAF=0.

---
# Variant Quality Score Recalibration (VQSR)

VQSR have 2 step.

    1. Create model with command `VariantRecalibrator`. Run separately for SNP and INDEL
    
    2. Apply model to data with command 'ApplyVQSR'

In [23]:
!gatk VariantRecalibrator -h

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -h
USAGE: VariantRecalibrator [arguments]

Build a recalibration model to score variant quality for filtering purposes
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output recal file used by ApplyVQSR  Required. 

--resource <FeatureInput>     A list of sites for which to apply a prior probability of being correct but which aren't
                              used by the algorithm (training and truth sets are required to run)  This argument must be
                              specified at least once. Required. 

--tranches-file <File>        The output tranches file us

In [24]:
!gatk ApplyVQSR -h

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -h
USAGE: ApplyVQSR [arguments]

Apply a score cutoff to filter variants based on a recalibration table
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output filtered and recalibrated VCF file in which each variant is annotated with its
                              VQSLOD value  Required. 

--recal-file <FeatureInput>   The input recal file used by ApplyVQSR  Required. 

--variant,-V <GATKPath>       One or more VCF files containing variants  This argument must be specified at least once.
                              Required. 


Optional Arguments:

--add-output-sam-program-recor

Create output directory

In [25]:
!mkdir -p ./output/vqsr

## Create model for SNP

Based on recommendation from GATK website

https://gatk.broadinstitute.org/hc/en-us/articles/4402736812443-Which-training-sets-arguments-should-I-use-for-running-VQSR

The standard set are store in

https://console.cloud.google.com/storage/browser/gcp-public-data--broad-references/hg38/v0?pageState=(%22StorageObjectListTable%22:(%22f%22:%22%255B%255D%22))

In [26]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz  \
    --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR  \
    -mode SNP \
    --max-gaussians 4 \
    -O ./output/vqsr/vqsr_snp.recal \
    --tranches-file ./output/vqsr/vqsr_snp.tranches

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQ -an MQ

## Create model for INDEL

In [ ]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 1 \
    -O ./output/vqsr/vqsr_indel.recal \
    --tranches-file ./output/vqsr/vqsr_indel.tranches

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP -mode INDEL --max-gaussians 1 -O ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches
14:19:04.127 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.co

## Apply VQSR SNP model

In [ ]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/genotypegvcf/combine.vcf \
  --recal-file ./output/vqsr/vqsr_snp.recal \
  -mode SNP \
  --tranches-file ./output/vqsr/vqsr_snp.tranches \
  --truth-sensitivity-filter-level 99.5 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_snp_recalibrated.vcf.gz

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/genotypegvcf/combine.vcf --recal-file ./output/vqsr/vqsr_snp.recal -mode SNP --tranches-file ./output/vqsr/vqsr_snp.tranches --truth-sensitivity-filter-level 99.5 --create-output-variant-index true -O ./output/vqsr/combine_snp_recalibrated.vcf.gz
14:19:24.208 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
14:19:24.324 INFO  ApplyVQSR - ------------------------------------------------------------
14:19:24.328 INFO  ApplyVQSR - The Ge

## Apply VQSR INDEL model

In [29]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/vqsr/combine_snp_recalibrated.vcf.gz \
  -mode INDEL \
  --recal-file ./output/vqsr/vqsr_indel.recal \
  --tranches-file ./output/vqsr/vqsr_indel.tranches \
  --truth-sensitivity-filter-level 99.0 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/vqsr/combine_snp_recalibrated.vcf.gz -mode INDEL --recal-file ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches --truth-sensitivity-filter-level 99.0 --create-output-variant-index true -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
14:19:30.613 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
14:19:30.715 INFO  ApplyVQSR - ------------------------------------------------------------
14:19:30.718 

### Gatk Hard filtering (Alternative)

Instead of using VQSR. GATK also provide a hard filtering setting.

https://gatk.broadinstitute.org/hc/en-us/articles/360035890471-Hard-filtering-germline-short-variants

In [30]:
"""
!gatk VariantFiltration \
     -V ./output/genotypegvcf/combine.vcf \
     -O ./output/genotypegvcf/combine_filtered_variants.vcf.gz \
     -filter "QD < 2.0" --filter-name "QD2" \
     -filter "MQ < 40.0" --filter-name "MQ40" \
     -filter "FS > 60.0" --filter-name "FS60" \
     -filter "SOR > 3.0" --filter-name "SOR3" \
     -filter "MQRankSum < -12.5" --filter-name "MQRankSum-12.5" \
     -filter "ReadPosRankSum < -8.0" --filter-name "ReadPosRankSum-8"
"""

'\n!gatk VariantFiltration      -V ./output/genotypegvcf/combine.vcf      -O ./output/genotypegvcf/combine_filtered_variants.vcf.gz      -filter "QD < 2.0" --filter-name "QD2"      -filter "MQ < 40.0" --filter-name "MQ40"      -filter "FS > 60.0" --filter-name "FS60"      -filter "SOR > 3.0" --filter-name "SOR3"      -filter "MQRankSum < -12.5" --filter-name "MQRankSum-12.5"      -filter "ReadPosRankSum < -8.0" --filter-name "ReadPosRankSum-8"\n'

## View your VCF file

In [31]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601430	.	A	C	857.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167;VQSLOD=-117.2;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467;VQSLOD=-42.05;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086;VQSLOD=-16.68;culprit=MQRankSum	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12

---

### Filter VCF with vcftools

In [32]:
!vcftools --gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz \
    --recode \
    --recode-INFO-all \
    --remove-filtered-all \
    --out ./output/vqsr/combine_indel_snp_recalibrated


VCFtools - 0.1.17
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
	--recode-INFO-all
	--out ./output/vqsr/combine_indel_snp_recalibrated
	--recode
	--remove-filtered-all

Using zlib version: 1.3.1
After filtering, kept 1 out of 1 Individuals
Outputting VCF file...
After filtering, kept 2671 out of a possible 2799 Sites
Run Time = 0.00 seconds


**BCFtools equivalent command**

In [ ]:
"""
!bcftools view \
    -m 2 -M 2 \
    -f PASS \
    -O v \
    -o ./output/vqsr/combine_indel_snp_recalibrated.vcf \
    ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
""""

### Check VCF file before and after filtering

In [33]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601430	.	A	C	857.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167;VQSLOD=-117.2;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467;VQSLOD=-42.05;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086;VQSLOD=-16.68;culprit=MQRankSum	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12

In [34]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf | head -10

chr14	18985420	.	T	C	54.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=0;DP=3;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=36.19;MQRankSum=-0.431;QD=18.21;ReadPosRankSum=0.967;SOR=1.179;VQSLOD=-0.0063;culprit=MQ	GT:AD:DP:GQ:PL	0/1:1,2:3:30:62,0,30
chr14	19180623	.	AT	A	286	PASS	AC=2;AF=1;AN=2;DP=8;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=42;QD=25.36;SOR=4.407;VQSLOD=7.47;culprit=QD	GT:AD:DP:GQ:PL	1/1:0,8:8:24:300,24,0
chr14	19180816	.	T	TG	212.93	PASS	AC=2;AF=1;AN=2;DP=6;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=34.87;QD=28.73;SOR=1.329;VQSLOD=2.25;culprit=SOR	GT:AD:DP:GQ:PL	1/1:0,6:6:18:227,18,0
chr14	19413462	.	G	C	102.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.401;DP=42;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=26.1;MQRankSum=0.742;QD=2.77;ReadPosRankSum=-1.689;SOR=0.064;VQSLOD=1.3;culprit=MQ	GT:AD:DP:GQ:PL	0/1:30,7:37:99:110,0,829
chr14	19433743	.	C	T	319.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.178;DP=42;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=26.87;MQRankSum=0.802;QD=8.2;ReadPosRankSum=-2.124;SOR=0.287;VQSLOD=2.13

---
**Compress and index VCF file**

In [35]:
!bgzip -c ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf > ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

In [36]:
!tabix -p vcf ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

---

## Exercise : Run the whole pipeline on sampleB

- Sequence QC
- Alignemnt
- Alignemnt post process
- Variant calling
- *Create combine VCF between sampleA and sampleB
- Variant filtering

---


In [40]:
%mkdir output_exercise/FastQC_Out

In [ ]:
# running fastqc on the B_1 and B_2 sequences
!fastqc -f fastq -t 2 -o ./output_exercise/FastQC_Out ./output_exercise/B_1.fq.gz ./output_exercise/B_2.fq.gz

application/gzip
application/gzip
Started analysis of B_1.fq.gz
Approx 5% complete for B_1.fq.gz
Approx 10% complete for B_1.fq.gz
Approx 15% complete for B_1.fq.gz
Started analysis of B_2.fq.gz
Approx 20% complete for B_1.fq.gz
Approx 5% complete for B_2.fq.gz
Approx 25% complete for B_1.fq.gz
Approx 10% complete for B_2.fq.gz
Approx 30% complete for B_1.fq.gz
Approx 15% complete for B_2.fq.gz
Approx 35% complete for B_1.fq.gz
Approx 20% complete for B_2.fq.gz
Approx 40% complete for B_1.fq.gz
Approx 25% complete for B_2.fq.gz
Approx 45% complete for B_1.fq.gz
Approx 30% complete for B_2.fq.gz
Approx 50% complete for B_1.fq.gz
Approx 35% complete for B_2.fq.gz
Approx 55% complete for B_1.fq.gz
Approx 40% complete for B_2.fq.gz
Approx 60% complete for B_1.fq.gz
Approx 45% complete for B_2.fq.gz
Approx 65% complete for B_1.fq.gz
Approx 50% complete for B_2.fq.gz
Approx 70% complete for B_1.fq.gz
Approx 55% complete for B_2.fq.gz
Approx 75% complete for B_1.fq.gz
Approx 60% complete for 

In [ ]:
# Checking that fastqc files have been created in the appropriate directory
%ls output_exercise/FastQC_Out/

B_1_fastqc.html  B_1_fastqc.zip  B_2_fastqc.html  B_2_fastqc.zip


In [ ]:
# making directory for output of fastp
mkdir output_exercise/fastp_output

In [49]:
# running fastp on the files generated by fastqc
!fastp -i ./data/FASTQ_Chr14/B_1.fq.gz \
    -I ./data/FASTQ_Chr14/B_2.fq.gz \
    -o ./output_exercise/fastp_output/clean_B_1.fq.gz \
    -O ./output_exercise/fastp_output/clean_B_2.fq.gz \
    -q 30 \
    -h ./output_exercise/fastp_output/B_fastp_report.html

Read1 before filtering:
total reads: 397602
total bases: 40157802
Q20 bases: 39898011(99.3531%)
Q30 bases: 38738731(96.4663%)
Q40 bases: 14612471(36.3876%)

Read2 before filtering:
total reads: 397602
total bases: 40157802
Q20 bases: 39623778(98.6702%)
Q30 bases: 38081683(94.8301%)
Q40 bases: 14121679(35.1655%)

Read1 after filtering:
total reads: 390677
total bases: 39449142
Q20 bases: 39248680(99.4918%)
Q30 bases: 38253784(96.9699%)
Q40 bases: 14566090(36.9237%)

Read2 after filtering:
total reads: 390677
total bases: 39449142
Q20 bases: 38987294(98.8293%)
Q30 bases: 37629533(95.3875%)
Q40 bases: 14086299(35.7075%)

Filtering result:
reads passed filter: 781354
reads failed due to low quality: 13270
reads failed due to too many N: 580
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 1190
bases trimmed due to adapters: 18554

Duplication rate: 0.000251508%

Insert size peak (evaluated by paired-end reads): 165

JSON report: fastp.json
H

In [50]:

!mkdir -p ./output_exercise/bwa

In [52]:
# running bwa to align the sequences generated by fastp
!bwa index ./reference/chr14.fa

!bwa mem -M -t 2 \
    -R "@RG\tID:A\tPL:ILLUMINA\tLB:lib1\tSM:A" \
    -o ./output_exercise/bwa/B.sam \
    ./reference/chr14.fa \
    ./output_exercise/fastp_output/clean_B_1.fq.gz \
    ./output_exercise/fastp_output/clean_B_2.fq.gz

[bwa_index] Pack FASTA... 0.44 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=214087436, availableWord=27063796
[BWTIncConstructFromPacked] 10 iterations done. 44642828 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 82473580 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 116093612 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 145971148 characters processed.
[BWTIncConstructFromPacked] 50 iterations done. 172522364 characters processed.
[BWTIncConstructFromPacked] 60 iterations done. 196117116 characters processed.
[bwt_gen] Finished constructing BWT in 69 iterations.
[bwa_index] 58.61 seconds elapse.
[bwa_index] Update BWT... 0.32 sec
[bwa_index] Pack forward-only FASTA... 0.30 sec
[bwa_index] Construct SA from BWT and Occ... 30.47 sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa index ./reference/chr14.fa
[main] Real time: 90.685 sec; CPU: 90.191 sec
[M::bwa_idx_load_from_disk] re

In [53]:
!samtools flagstat -@ 2 ./output_exercise/bwa/B.sam

781493 + 0 in total (QC-passed reads + QC-failed reads)
781354 + 0 primary
139 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
781488 + 0 mapped (100.00% : N/A)
781349 + 0 primary mapped (100.00% : N/A)
781354 + 0 paired in sequencing
390677 + 0 read1
390677 + 0 read2
780190 + 0 properly paired (99.85% : N/A)
781344 + 0 with itself and mate mapped
5 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


In [54]:
# conversion of SAM to BAM
!samtools view -bS \
    -@ 2 \
    -o ./output_exercise/bwa/B.bam \
    ./output_exercise/bwa/B.sam

In [56]:
# sorting, indexing, and filtering unmapped reads on BAM file 

!samtools sort -@ 2 \
    -o ./output_exercise/bwa/B_sort.bam \
    ./output_exercise/bwa/B.bam

!samtools index -@ 2 ./output_exercise/bwa/B_sort.bam

!samtools view -b \
    -F 4 -F 8 \
    -@ 2 \
    -o ./output_exercise/bwa/B_sort_map.bam \
    ./output_exercise/bwa/B_sort.bam

[bam_sort_core] merging from 0 files and 2 in-memory blocks...


In [57]:
!samtools flagstat ./output_exercise/bwa/B_sort.bam

781493 + 0 in total (QC-passed reads + QC-failed reads)
781354 + 0 primary
139 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
781488 + 0 mapped (100.00% : N/A)
781349 + 0 primary mapped (100.00% : N/A)
781354 + 0 paired in sequencing
390677 + 0 read1
390677 + 0 read2
780190 + 0 properly paired (99.85% : N/A)
781344 + 0 with itself and mate mapped
5 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


In [ ]:
# Comparing BAM flags after filtering, with 0 singletons as they have been filtered out
!samtools flagstat ./output_exercise/bwa/B_sort_map.bam

781483 + 0 in total (QC-passed reads + QC-failed reads)
781344 + 0 primary
139 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
781483 + 0 mapped (100.00% : N/A)
781344 + 0 primary mapped (100.00% : N/A)
781344 + 0 paired in sequencing
390672 + 0 read1
390672 + 0 read2
780190 + 0 properly paired (99.85% : N/A)
781344 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


In [59]:
# marking duplicates
!gatk --java-options "-Xmx4G" MarkDuplicates \
        -I ./output_exercise/bwa/B_sort.bam \
        -O ./output_exercise/bwa/B_sort_markdup.bam \
        -M ./output_exercise/bwa/B_sort_markdup_metrics.txt \
        --TMP_DIR .

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar MarkDuplicates -I ./output_exercise/bwa/B_sort.bam -O ./output_exercise/bwa/B_sort_markdup.bam -M ./output_exercise/bwa/B_sort_markdup_metrics.txt --TMP_DIR .
15:06:36.973 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Fri Apr 24 15:06:37 CEST 2026] MarkDuplicates --INPUT ./output_exercise/bwa/B_sort.bam --OUTPUT ./output_exercise/bwa/B_sort_markdup.bam --METRICS_FILE ./output_exercise/bwa/B_sort_markdup_metrics.txt --TMP_DIR . --MAX_SEQUENCES_FOR_DISK_

In [61]:
!gatk --java-options "-Xmx4G" CollectWgsMetrics \
        -MQ 20 -Q 20 \
        -R ./reference/chr14.fa \
        -O ./output_exercise/bwa/B_sort_markdup_WGSmetrics.txt \
        -I ./output_exercise/bwa/B_sort_markdup.bam

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar CollectWgsMetrics -MQ 20 -Q 20 -R ./reference/chr14.fa -O ./output_exercise/bwa/B_sort_markdup_WGSmetrics.txt -I ./output_exercise/bwa/B_sort_markdup.bam
15:07:36.817 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Fri Apr 24 15:07:36 CEST 2026] CollectWgsMetrics --INPUT ./output_exercise/bwa/B_sort_markdup.bam --OUTPUT ./output_exercise/bwa/B_sort_markdup_WGSmetrics.txt --MINIMUM_MAPPING_QUALITY 20 --MINIMUM_BASE_QUALITY 20 --REFERENCE_SEQUENCE ./refere

In [ ]:
# Verification of index on reference and creating an output directory for BQSR
!samtools faidx ./reference/chr14.fa

!gatk --java-options "-Xmx4G" CreateSequenceDictionary \
    -R ./reference/chr14.fa

!mkdir -p ./output_exercise/bqsr

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary -R ./reference/chr14.fa
INFO	2026-04-24 15:08:57	CreateSequenceDictionary	Output dictionary will be written in /home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/./reference/chr14.dict
15:08:57.932 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Fri Apr 24 15:08:57 CEST 2026] CreateSequenceDictionary --REFERENCE ./reference/chr14.fa --TRUNC

In [ ]:
# Building and applying BQSR model for base quality score recallibration

!gatk --java-options "-Xmx4G" BaseRecalibrator \
        -R ./reference/chr14.fa \
        -I ./output_exercise/bwa/B_sort_markdup.bam \
        -O ./output_exercise/bqsr/B_sort_markdup_BQSR.report \
        --known-sites ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
        --known-sites ./db/resources_broad_hg38_v0_Homo_sapiens_assembly38.known_indels.vcf.gz \
        --known-sites ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz

!gatk --java-options "-Xmx4G" ApplyBQSR \
        -I ./output_exercise/bwa/B_sort_markdup.bam \
        -O ./output_exercise/bqsr/B_sort_markdup_recal.bam \
        --bqsr-recal-file ./output_exercise/bqsr/B_sort_markdup_BQSR.report

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar BaseRecalibrator -R ./reference/chr14.fa -I ./output_exercise/bwa/B_sort_markdup.bam -O ./output_exercise/bqsr/B_sort_markdup_BQSR.report --known-sites ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz --known-sites ./db/resources_broad_hg38_v0_Homo_sapiens_assembly38.known_indels.vcf.gz --known-sites ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
15:11:17.043 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:11:17.139 INFO 

In [64]:
!mkdir -p ./output_exercise/haplotypecaller
!mkdir -p ./tmp

In [ ]:
# gatk haplotype caller, with input .bam (filtered, with marked duplicates) and variants saved as .vcf

!gatk --java-options "-Xmx8G" HaplotypeCaller \
    -R ./reference/chr14.fa \
    -I ./output_exercise/bqsr/B_sort_markdup_recal.bam \
    -O ./output_exercise/haplotypecaller/B.g.vcf \
    -ERC GVCF \
    --tmp-dir ./tmp

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R ./reference/chr14.fa -I ./output_exercise/bqsr/B_sort_markdup_recal.bam -O ./output_exercise/haplotypecaller/B.g.vcf -ERC GVCF --tmp-dir ./tmp
15:15:54.192 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:15:54.310 INFO  HaplotypeCaller - ------------------------------------------------------------
15:15:54.313 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
15:15:54.313 INFO  HaplotypeCaller - For support and docu

In [ ]:
# preparation of name mapping file
# have for both A and for B sample - maybe just copy the A.g.vcf file to the output_exercise/haplotypecaller folder

!cp ./output/haplotypecaller/A.g.vcf ./output_exercise/haplotypecaller/
!cp ./output/haplotypecaller/A.g.vcf.idx ./output_exercise/haplotypecaller/

!ls ./output_exercise/haplotypecaller/*.g.vcf | xargs -n1 basename \
    | awk '{split($0,a,".");print a[1]}' > ./output_exercise/haplotypecaller/input_name.txt

!realpath ./output_exercise/haplotypecaller/*.g.vcf > ./output_exercise/haplotypecaller/input_path.txt

!paste ./output_exercise/haplotypecaller/input_name.txt \
    ./output_exercise/haplotypecaller/input_path.txt  > ./output_exercise/haplotypecaller/samples_map.txt

In [93]:
!cat ./output_exercise/haplotypecaller/samples_map.txt

A	/home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output_exercise/haplotypecaller/A.g.vcf
B	/home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output_exercise/haplotypecaller/B.g.vcf


In [ ]:
# creating genome DB

!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-workspace-path ./output_exercise/genomeDB2 \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./output_exercise/haplotypecaller/samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xms8G -Xmx16G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport --genomicsdb-workspace-path ./output_exercise/genomeDB2 --batch-size 50 -L chr14 --sample-name-map ./output_exercise/haplotypecaller/samples_map.txt --tmp-dir ./tmp --reader-threads 2
16:20:28.792 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
16:20:28.899 INFO  GenomicsDBImport - ------------------------------------------------------------
16:20:28.903 INFO  GenomicsDBImport - The Genome Analysis Toolkit (GATK) v4.6.2.0
16:20:28

In [95]:
!mkdir -p ./output_exercise/genotypegvcf

In [126]:
# multiple sample genotyping

!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V gendb://${PWD}/output_exercise/genomeDB2 \
    -O ./output_exercise/genotypegvcf/combine.vcf \
    --tmp-dir ./tmp


Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V gendb:///home/mmilenkov/Documents/Informatics/GIT/UZH-BIO392-FS26/course-results/martin_milenkov/2026-04-24/2026-04-24_BIO0392_FS26_notebook/notebook/output_exercise/genomeDB2 -O ./output_exercise/genotypegvcf/combine.vcf --tmp-dir ./tmp
16:20:44.094 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
16:20:44.188 INFO  GenotypeGVCFs - ------------------------------------------------------------
16:20:44.191 INFO  Geno

In [127]:
!bcftools view -h ./output_exercise/genotypegvcf/combine.vcf | tail

##INFO=<ID=RAW_MQandDP,Number=2,Type=Integer,Description="Raw data (sum of squared MQ and total depth) for improved RMS Mapping Quality calculation. Incompatible with deprecated RAW_MQ formulation.">
##INFO=<ID=ReadPosRankSum,Number=1,Type=Float,Description="Z-score from Wilcoxon rank sum test of Alt vs. Ref read position bias">
##INFO=<ID=SOR,Number=1,Type=Float,Description="Symmetric Odds Ratio of 2x2 contingency table to detect strand bias">
##contig=<ID=chr14,length=107043718>
##source=GenomicsDBImport
##source=GenotypeGVCFs
##source=HaplotypeCaller
##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output_exercise/genotypegvcf/combine.vcf; Date=Fri Apr 24 16:20:57 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	A	B


In [128]:
!bcftools view -H ./output_exercise/genotypegvcf/combine.vcf | head -5

chr14	18601385	.	C	T	68.94	.	AC=1;AF=0.25;AN=4;BaseQRankSum=-1.123;DP=172;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.25;MQ=35.53;MQRankSum=-0.143;QD=1.21;ReadPosRankSum=-0.706;SOR=4.167	GT:AD:DP:GQ:PL	0/0:111,0:111:18:0,18,3445	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	1495.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.495;DP=202;ExcessHet=1.7609;FS=175.704;MLEAC=2;MLEAF=0.5;MQ=33.4;MQRankSum=0.693;QD=8;ReadPosRankSum=-0.132;SOR=8.222	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	7511.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.011;DP=640;ExcessHet=1.7609;FS=29.013;MLEAC=2;MLEAF=0.5;MQ=37.94;MQRankSum=-4.714;QD=12.91;ReadPosRankSum=-2.075;SOR=2.328	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	2797.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-3.946;DP=216;ExcessHet=1.7609;FS=21.535;MLEAC=2;MLEAF=0.5;MQ=44.78;MQRankSum=-8.252;QD=13.51;ReadPosRankSum=0.06;SOR=4.426	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919	0/1:48,42:

In [105]:
!mkdir -p ./output_exercise/vqsr

In [129]:
#SNP Variant control model

!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output_exercise/genotypegvcf/combine.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz  \
    --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR  \
    -mode SNP \
    --max-gaussians 4 \
    -O ./output_exercise/vqsr/vqsr_snp.recal \
    --tranches-file ./output_exercise/vqsr/vqsr_snp.tranches

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output_exercise/genotypegvcf/combine.vcf --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an 

In [130]:
#INDEL Variant control model

!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output_exercise/genotypegvcf/combine.vcf \
    --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 1 \
    -O ./output_exercise/vqsr/vqsr_indel.recal \
    --tranches-file ./output_exercise/vqsr/vqsr_indel.tranches

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output_exercise/genotypegvcf/combine.vcf --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP -mode INDEL --max-gaussians 1 -O ./output_exercise/vqsr/vqsr_indel.recal --tranches-file ./output_exercise/vqsr/vqsr_indel.tranches
16:21:14.954 INFO  NativeLibraryLoader - Loading libgkl_compression.so from j

In [131]:
# Applying VSQR SNP model

!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output_exercise/genotypegvcf/combine.vcf \
  --recal-file ./output_exercise/vqsr/vqsr_snp.recal \
  -mode SNP \
  --tranches-file ./output_exercise/vqsr/vqsr_snp.tranches \
  --truth-sensitivity-filter-level 99.5 \
  --create-output-variant-index true \
  -O ./output_exercise/vqsr/combine_snp_recalibrated.vcf.gz

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output_exercise/genotypegvcf/combine.vcf --recal-file ./output_exercise/vqsr/vqsr_snp.recal -mode SNP --tranches-file ./output_exercise/vqsr/vqsr_snp.tranches --truth-sensitivity-filter-level 99.5 --create-output-variant-index true -O ./output_exercise/vqsr/combine_snp_recalibrated.vcf.gz
16:21:20.044 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
16:21:20.134 INFO  ApplyVQSR - ------------------------------------------------------------
1

In [132]:
# Applying INDEL model

!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output_exercise/vqsr/combine_snp_recalibrated.vcf.gz \
  -mode INDEL \
  --recal-file ./output_exercise/vqsr/vqsr_indel.recal \
  --tranches-file ./output_exercise/vqsr/vqsr_indel.tranches \
  --truth-sensitivity-filter-level 99.0 \
  --create-output-variant-index true \
  -O ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz

Using GATK jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output_exercise/vqsr/combine_snp_recalibrated.vcf.gz -mode INDEL --recal-file ./output_exercise/vqsr/vqsr_indel.recal --tranches-file ./output_exercise/vqsr/vqsr_indel.tranches --truth-sensitivity-filter-level 99.0 --create-output-variant-index true -O ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz
16:21:24.101 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/home/mmilenkov/.conda/envs/human_env/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
16:21:24.199 INFO  ApplyVQSR - --------------------------------------

In [133]:
# Preview
!bcftools view -H ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601385	.	C	T	68.94	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.25;AN=4;BaseQRankSum=-1.123;DP=172;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.25;MQ=35.53;MQRankSum=-0.143;QD=1.21;ReadPosRankSum=-0.706;SOR=4.167;VQSLOD=-37.63;culprit=FS	GT:AD:DP:GQ:PL	0/0:111,0:111:18:0,18,3445	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	1495.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.495;DP=202;ExcessHet=1.7609;FS=175.704;MLEAC=2;MLEAF=0.5;MQ=33.4;MQRankSum=0.693;QD=8;ReadPosRankSum=-0.132;SOR=8.222;VQSLOD=-1222;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	7511.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.011;DP=640;ExcessHet=1.7609;FS=29.013;MLEAC=2;MLEAF=0.5;MQ=37.94;MQRankSum=-4.714;QD=12.91;ReadPosRankSum=-2.075;SOR=2.328;VQSLOD=-47.83;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	2797.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=

In [134]:
# Filtering with VCFtools

!vcftools --gzvcf ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz \
    --recode \
    --recode-INFO-all \
    --remove-filtered-all \
    --out ./output_exercise/vqsr/combine_indel_snp_recalibrated


VCFtools - 0.1.17
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--gzvcf ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz
	--recode-INFO-all
	--out ./output_exercise/vqsr/combine_indel_snp_recalibrated
	--recode
	--remove-filtered-all

Using zlib version: 1.3.1
After filtering, kept 2 out of 2 Individuals
Outputting VCF file...
After filtering, kept 2849 out of a possible 2926 Sites
Run Time = 0.00 seconds


In [135]:
!bcftools view -H ./output_exercise/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601385	.	C	T	68.94	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.25;AN=4;BaseQRankSum=-1.123;DP=172;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.25;MQ=35.53;MQRankSum=-0.143;QD=1.21;ReadPosRankSum=-0.706;SOR=4.167;VQSLOD=-37.63;culprit=FS	GT:AD:DP:GQ:PL	0/0:111,0:111:18:0,18,3445	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	1495.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.495;DP=202;ExcessHet=1.7609;FS=175.704;MLEAC=2;MLEAF=0.5;MQ=33.4;MQRankSum=0.693;QD=8;ReadPosRankSum=-0.132;SOR=8.222;VQSLOD=-1222;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	7511.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.011;DP=640;ExcessHet=1.7609;FS=29.013;MLEAC=2;MLEAF=0.5;MQ=37.94;MQRankSum=-4.714;QD=12.91;ReadPosRankSum=-2.075;SOR=2.328;VQSLOD=-47.83;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	2797.5	VQSRTrancheSNP99.90to100.00	AC=2;AF=0.5;AN=4;BaseQRankSum=

In [136]:
!bcftools view -H ./output_exercise/vqsr/combine_indel_snp_recalibrated.recode.vcf | head -10

chr14	18999581	.	C	G	1799.5	PASS	AC=2;AF=0.5;AN=4;BaseQRankSum=-2.938;DP=199;ExcessHet=1.7609;FS=0;MLEAC=2;MLEAF=0.5;MQ=39.01;MQRankSum=1.48;QD=9.18;ReadPosRankSum=-0.466;SOR=0.138;VQSLOD=-8.57;culprit=MQ	GT:AD:DP:GQ:PGT:PID:PL:PS	0|1:72,36:108:99:0|1:18999581_C_G:1288,0,2896:18999581	0|1:70,18:88:99:0|1:18999581_C_G:521,0,2865:18999581
chr14	18999587	.	T	C	1765.5	PASS	AC=2;AF=0.5;AN=4;BaseQRankSum=-6.375;DP=199;ExcessHet=1.7609;FS=0;MLEAC=2;MLEAF=0.5;MQ=38.72;MQRankSum=1.59;QD=8.87;ReadPosRankSum=-0.007;SOR=0.127;VQSLOD=-9.744;culprit=MQ	GT:AD:DP:GQ:PGT:PID:PL:PS	0|1:74,36:110:99:0|1:18999581_C_G:1282,0,2986:18999581	0|1:72,17:89:99:0|1:18999581_C_G:493,0,2951:18999581
chr14	19180623	.	AT	A	545.99	PASS	AC=4;AF=1;AN=4;DP=15;ExcessHet=0;FS=0;MLEAC=4;MLEAF=1;MQ=39.43;QD=25.36;SOR=5.549;VQSLOD=3.48;culprit=QD	GT:AD:DP:GQ:PL	1/1:0,8:8:24:300,24,0	1/1:0,7:7:21:262,21,0
chr14	19180816	.	T	TG	412.74	PASS	AC=4;AF=1;AN=4;DP=11;ExcessHet=0;FS=0;MLEAC=4;MLEAF=1;MQ=33.82;QD=28.73;SOR=2.494;VQSLOD=